In [ ]:
from __future__ import annotations

%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
from ipywidgets import widgets

from latent_dynamics.dayang.activations import Activations, extract_activations
from latent_dynamics.dayang.analysis import compute_per_layer, compute_per_token
from latent_dynamics.dayang.data import DATASET_REGISTRY, load_dataset_from_spec
from latent_dynamics.dayang.model import (
    MODEL_REGISTRY,
    get_token_embeddings,
    get_token_groups,
    load_model_and_tokenizer,
    print_token_groups,
)
from latent_dynamics.dayang.utils import Cache, select
from latent_dynamics.dayang.visualizations import (
    plot_explained_variance,
    plot_per_layer,
    plot_per_token,
    plot_token_embeddings,
)
from latent_dynamics.dayang.widgets import (
    ActivationsExtractorWidget,
    ActivationsSelectorWidget,
    TokenEmbeddingsLoaderWidget,
    ReadersSelectorWidget,
)

torch.set_grad_enabled(False)

print(f"Available models:{'\n - '.join([''] + list(MODEL_REGISTRY))}")
print(f"Available datasets:{'\n - '.join([''] + list(DATASET_REGISTRY))}")

# Interactive analysis

In [ ]:
def analyze_per_token():
    CACHE = Cache(maxsize=10, encoders={Activations: lambda x: (sorted(x.samples.index.tolist()), sorted(x.layers))})

    # Define layer 1 widgets for extracting activations
    out_layer1 = widgets.Output()
    w_act_extractor = ActivationsExtractorWidget(
        out=out_layer1,
        search_path="../activations",
        dataset="xstest",
    )
    w_emb_loader = TokenEmbeddingsLoaderWidget(
        out=out_layer1,
    )
    layer1 = widgets.VBox([w_act_extractor, w_emb_loader, out_layer1])

    # Define layer 2 widgets for analyzing activations
    w_dim_selector = widgets.Dropdown(
        options=["per token", "per layer"], value="per token", description="Dimension:"
    )
    w_readers_selector = ReadersSelectorWidget()
    w_train_selector = ActivationsSelectorWidget(
        pool_method="all",
    )
    w_train_separate = widgets.Checkbox(value=False, description="Separate analysis")
    w_train_link = widgets.Checkbox(value=True, description="Same as evaluation")
    w_eval_selector = ActivationsSelectorWidget(
        pool_method="all",
    )
    w_eval_separate = widgets.Checkbox(value=False, description="Separate plots")
    btn_analyze = widgets.Button(description="Analyze", button_style="primary", disabled=True)
    out_analyze = widgets.Output()

    col_train = widgets.VBox([widgets.HTML("<b>PCA</b>"), w_train_selector, w_train_separate, w_train_link])
    col_eval = widgets.VBox([widgets.HTML("<b>Visualization</b>"), w_eval_selector, w_eval_separate])
    layer2 = widgets.VBox(
        [
            w_dim_selector,
            w_readers_selector,
            widgets.HBox([col_train, col_eval]),
            btn_analyze,
            out_analyze,
        ]
    )

    # Register handlers for layer 1
    def update_layer2_widgets(*args):
        CACHE.clear()

        # Update widgets
        activations = w_act_extractor.activations
        if activations is None:
            btn_analyze.disabled = True
        else:
            w_train_selector.set_activations(activations)
            w_eval_selector.set_activations(activations)
            btn_analyze.disabled = False
        out_analyze.clear_output()

    w_act_extractor.observe(update_layer2_widgets, names="value")
    update_layer2_widgets()

    # Register handlers for layer 2
    def update_link(*args):
        if w_train_link.value:
            w_train_selector.link(w_eval_selector)
        else:
            w_train_selector.unlink()

    w_train_link.observe(update_link, names="value")
    update_link()

    def do_plot(*args):
        activations = w_act_extractor.activations
        if activations is None:
            return

        with out_analyze:
            out_analyze.clear_output(wait=True)

            if w_dim_selector.value == "per layer":
                compute_fn = compute_per_layer
                plot_fn = plot_per_layer
            else:
                compute_fn = compute_per_token
                plot_fn = plot_per_token

            # Train and cache model
            train_activations = activations.select(
                sample_ids=w_train_selector.samples or None,
                layers=w_train_selector.layers or None,
            )
            readers = CACHE(compute_fn)(
                train_activations,
                w_readers_selector.readers,
                pool_method=w_train_selector.pool_method,
                exclude_bos=w_train_selector.exclude_bos,
                exclude_special_tokens=w_train_selector.exclude_special_tokens,
                separate=w_train_separate.value,
            )

            # Plot explained variance
            plot_explained_variance(readers)

            # Plot evaluation
            eval_activations = activations.select(
                sample_ids=w_eval_selector.samples or None,
                layers=w_eval_selector.layers or None,
            )
            plot_fn(
                eval_activations,
                readers,
                pool_method=w_eval_selector.pool_method,
                exclude_bos=w_eval_selector.exclude_bos,
                exclude_special_tokens=w_eval_selector.exclude_special_tokens,
                token_embeddings=w_emb_loader.token_embeddings,
                separate=w_eval_separate.value,
            )

    btn_analyze.on_click(do_plot)

    # Display widgets
    display(widgets.VBox([layer1, widgets.HTML("<hr>"), layer2]))


analyze_per_token()

# Manual analysis

In [ ]:
dataset = load_dataset_from_spec("xstest", max_samples=200)

In [ ]:
model, tokenizer = load_model_and_tokenizer("gemma3_270m")

## Generate responses

In [ ]:
from latent_dynamics.dayang.activations import _prepare_sample


def generate(model, tokenizer, sample, include_response, apply_chat_template):
    sample_prepared = _prepare_sample(
        sample, tokenizer, include_response=include_response, apply_chat_template=apply_chat_template
    )
    inputs = tokenizer(sample_prepared["input"], return_tensors="pt", add_special_tokens=not apply_chat_template)
    outputs = model.generate(**inputs, max_new_tokens=10)
    response = tokenizer.decode(outputs)[0]
    return response


def test_generate(model, tokenizer, sample, include_responses=[False, "Sure", "Sorry"], apply_chat_template=True):
    print(f"ID:             {sample['id']}")
    print(f"is_safe:        {sample['is_safe']}")
    print(f"is_adversarial: {sample['is_adversarial']}")
    for include_response in include_responses:
        print("-" * 50)
        response = generate(
            model,
            tokenizer,
            sample,
            include_response=include_response,
            apply_chat_template=apply_chat_template,
        )
        print(response)


test_generate(model, tokenizer, dataset[0], apply_chat_template=True)

## Extract activations

In [ ]:
activations = extract_activations(model, tokenizer, dataset, apply_chat_template=True)
activations.samples

In [ ]:
activations = Activations.load("../activations/xstest/gemma3_270m/chat")
activations.samples

In [ ]:
activations.extract_topk(model, tokenizer, layer=18)

## Token embedding analysis

In [ ]:
token_groups = get_token_groups(tokenizer)
print_token_groups(tokenizer, token_groups)

In [ ]:
plot_token_embeddings(
    model,
    tokenizer,
    token_groups,
    num_components=8,
)

## PCA analysis per layer

In [ ]:
readers = compute_per_layer(
    activations,
    ["pca", "pca"],
    pool_method="last",
)
plot_explained_variance(readers)

In [ ]:
plot_per_layer(
    activations.select(activations.samples_safe.index[:3].tolist() + activations.samples_unsafe.index[:3].tolist()),
    readers,
    pool_method=slice(-6, None),
    exclude_special_tokens=False,
)

## PCA analysis per token

In [ ]:
readers = compute_per_token(
    activations.select(activations.samples_safe.index[:3].tolist() + activations.samples_unsafe.index[:3].tolist()),
    readers=["pca", "pca"],
    pool_method=slice(-8, None),
    separate=True,
)
plot_explained_variance(readers)

In [ ]:
plot_per_token(
    activations.select(activations.samples_safe.index[:3].tolist() + activations.samples_unsafe.index[:3].tolist()),
    readers,
    pool_method=slice(-8, None),
    separate=True,
    token_embeddings=get_token_embeddings(model, tokenizer, select(range(tokenizer.vocab_size), at_most=0.25)),
)